In [1]:
import pandas as pd, numpy as np
from lsff_utils import data_processing

In [2]:
directory = "/snfs1/DATA/DHS_PROG_DHS/IND/2015_2016/"
results_dir = "../results"

### WRA

In [3]:
%%time

wra_columns = {
    "v001": "cluster_number",
    "v002": "household_number",
    "v003": "line_number",
    "v005": "weight",
    "v008": "interview_date",
    "v011": "date_of_birth",
    "v190": "wealth_quintile",
}
wra_data = pd.read_stata(
    directory + "IND_DHS7_2015_2016_WN_IAIR74FL_Y2018M12D06.DTA",
    columns=wra_columns.keys(),
)

CPU times: user 25.5 s, sys: 13.7 s, total: 39.3 s
Wall time: 47.1 s


In [4]:
wra_data = wra_data[wra_columns.keys()].rename(columns=wra_columns)

In [5]:
wra_data["wealth_quintile"] = data_processing.recode_dhs_wealth_quintile(
    wra_data.wealth_quintile
)

In [6]:
wra_data["weight"] = wra_data.weight / 1_000_000

### Adult mortality

Adult mortality included in HH for India (recent household members who died).

In [7]:
%%time

household_columns = {
    "hv005": "weight",
    "sh70": "any_died",
    "sh71": "num_died",
    "hv270": "wealth_quintile",
}
MAX_NUM_DEATHS = 5
death_columns = {
    "sh73": "sex",
    "sh74u": "age_at_death_unit",
    "sh74n": "age_at_death",
    "sh75m": "month_of_death",
    "sh75y": "year_of_death",
    "sh76": "death_violence_or_accident",
    "sh77": "death_during_pregnancy_or_childbirth",
}
columns = list(household_columns.keys())
for death_num in range(1, MAX_NUM_DEATHS + 1):
    columns += [c + "_" + str(death_num) for c in death_columns.keys()]

adult_mortality_data = pd.read_stata(
    directory + "IND_DHS7_2015_2016_HH_IAHR74FL_Y2018M12D06.DTA", columns=columns
)
adult_mortality_data

CPU times: user 24.4 s, sys: 10 s, total: 34.4 s
Wall time: 42 s


,hv005,sh70,sh71,hv270,sh73_1,sh74u_1,sh74n_1,sh75m_1,sh75y_1,sh76_1,...,sh75y_4,sh76_4,sh77_4,sh73_5,sh74u_5,sh74n_5,sh75m_5,sh75y_5,sh76_5,sh77_5
0,191072,no,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,191072,no,NaN,richer,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,191072,no,NaN,richer,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,191072,no,NaN,richer,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,191072,no,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
601504,2270734,no,NaN,poorest,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
601505,2270734,no,NaN,poorer,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
601506,2270734,no,NaN,richer,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
601507,2270734,no,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# inspired by https://stackoverflow.com/a/67393747/
adult_mortality_data_reshaped = adult_mortality_data[
    [c for c in adult_mortality_data.columns if c.split("_")[0] in death_columns.keys()]
].copy()
adult_mortality_data_reshaped.columns = adult_mortality_data_reshaped.columns.str.split(
    "_", expand=True
)
adult_mortality_data_reshaped

,sh73,sh74u,sh74n,sh75m,sh75y,sh76,sh77,sh73,sh74u,sh74n,...,sh75y,sh76,sh77,sh73,sh74u,sh74n,sh75m,sh75y,sh76,sh77
,1,1,1,1,1,1,1,2,2,2,...,4,4,4,5,5,5,5,5,5,5
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
601504,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
601505,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
601506,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
adult_mortality_data_reshaped[list(household_columns.keys())] = adult_mortality_data[
    list(household_columns.keys())
]
adult_mortality_data_reshaped

,sh73,sh74u,sh74n,sh75m,sh75y,sh76,sh77,sh73,sh74u,sh74n,...,sh74u,sh74n,sh75m,sh75y,sh76,sh77,hv005,sh70,sh71,hv270
,1,1,1,1,1,1,1,2,2,2,...,5,5,5,5,5,5,,,,
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,191072,no,NaN,middle
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,191072,no,NaN,richer
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,191072,no,NaN,richer
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,191072,no,NaN,richer
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,191072,no,NaN,middle
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
601504,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2270734,no,NaN,poorest
601505,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2270734,no,NaN,poorer
601506,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2270734,no,NaN,richer


In [10]:
# Get a row per death
adult_mortality_data_reshaped = (
    adult_mortality_data_reshaped.set_index(list(household_columns.keys()))
    .swaplevel(axis=1)
    .stack(0)
    .reset_index()
    .drop(columns=[f"level_{len(household_columns)}"])
)
adult_mortality_data_reshaped

,hv005,sh70,sh71,hv270,sh73,sh74n,sh74u,sh75m,sh75y,sh76,sh77
0,8939,yes,1.0,middle,male,51.0,year,april,2015.0,no,NaN
1,8939,yes,1.0,middle,male,56.0,months,april,2015.0,yes,NaN
2,4495,yes,1.0,middle,male,45.0,year,august,2014.0,no,NaN
3,4495,yes,2.0,richer,male,35.0,year,may,2014.0,no,NaN
4,4495,yes,2.0,richer,male,40.0,year,august,2014.0,no,NaN
...,...,...,...,...,...,...,...,...,...,...,...
74940,2463153,yes,2.0,poorer,female,21.0,year,september,2013.0,yes,NaN
74941,2270734,yes,1.0,middle,female,80.0,year,november,2014.0,no,no
74942,2270734,yes,1.0,middle,male,50.0,year,january,2015.0,no,NaN
74943,2270734,yes,1.0,middle,male,45.0,months,january,2012.0,yes,NaN


In [11]:
adult_mortality_data = (
    adult_mortality_data_reshaped[
        list(household_columns.keys()) + list(death_columns.keys())
    ]
    .rename(columns=household_columns)
    .rename(columns=death_columns)
)
adult_mortality_data["wealth_quintile"] = data_processing.recode_dhs_wealth_quintile(
    adult_mortality_data.wealth_quintile
)
adult_mortality_data["weight"] = adult_mortality_data.weight / 1_000_000
adult_mortality_data

,weight,any_died,num_died,wealth_quintile,sex,age_at_death_unit,age_at_death,month_of_death,year_of_death,death_violence_or_accident,death_during_pregnancy_or_childbirth
0,0.008939,yes,1.0,3,male,year,51.0,april,2015.0,no,NaN
1,0.008939,yes,1.0,3,male,months,56.0,april,2015.0,yes,NaN
2,0.004495,yes,1.0,3,male,year,45.0,august,2014.0,no,NaN
3,0.004495,yes,2.0,4,male,year,35.0,may,2014.0,no,NaN
4,0.004495,yes,2.0,4,male,year,40.0,august,2014.0,no,NaN
...,...,...,...,...,...,...,...,...,...,...,...
74940,2.463153,yes,2.0,2,female,year,21.0,september,2013.0,yes,NaN
74941,2.270734,yes,1.0,3,female,year,80.0,november,2014.0,no,no
74942,2.270734,yes,1.0,3,male,year,50.0,january,2015.0,no,NaN
74943,2.270734,yes,1.0,3,male,months,45.0,january,2012.0,yes,NaN


### Births

In [12]:
birth_columns = {
    "v005": "weight",
    "v008": "interview_date",
    "v190": "wealth_quintile",
    "b3": "birth_date",
    "m18": "size_of_child",
    "m19": "birth_weight_kilograms",
    "s220a": "duration_of_pregnancy",
}

birth_data = pd.read_stata(
    directory + "IND_DHS7_2015_2016_BR_IABR74FL_Y2018M12D06.DTA",
    columns=birth_columns.keys(),
)
birth_data

,v005,v008,v190,b3,m18,m19,s220a
0,191760,1387,middle,1141,NaN,NaN,NaN
1,191760,1387,middle,1117,NaN,NaN,NaN
2,191760,1387,middle,1089,NaN,NaN,NaN
3,191760,1387,richer,1154,NaN,NaN,NaN
4,191760,1387,richer,1129,NaN,NaN,NaN
...,...,...,...,...,...,...,...
1315612,2380715,1385,richer,1346,very large,3250.0,9.0
1315613,2380715,1385,middle,1100,NaN,NaN,NaN
1315614,2380715,1385,middle,1059,NaN,NaN,NaN
1315615,2380715,1385,middle,1027,NaN,NaN,NaN


In [13]:
birth_data = birth_data[birth_columns.keys()].rename(columns=birth_columns)
birth_data["wealth_quintile"] = data_processing.recode_dhs_wealth_quintile(
    birth_data.wealth_quintile
)
birth_data["weight"] = birth_data.weight / 1_000_000
birth_data

,weight,interview_date,wealth_quintile,birth_date,size_of_child,birth_weight_kilograms,duration_of_pregnancy
0,0.191760,1387,3,1141,NaN,NaN,NaN
1,0.191760,1387,3,1117,NaN,NaN,NaN
2,0.191760,1387,3,1089,NaN,NaN,NaN
3,0.191760,1387,4,1154,NaN,NaN,NaN
4,0.191760,1387,4,1129,NaN,NaN,NaN
...,...,...,...,...,...,...,...
1315612,2.380715,1385,4,1346,very large,3250.0,9.0
1315613,2.380715,1385,3,1100,NaN,NaN,NaN
1315614,2.380715,1385,3,1059,NaN,NaN,NaN
1315615,2.380715,1385,3,1027,NaN,NaN,NaN


### Household members

In [14]:
%%time

hhm_columns = {
    "hv001": "cluster_number",
    "hv002": "household_number",
    "hv005": "weight",
    "hv008": "date_of_interview",
    "hvidx": "line_number",
    "hml18": "currently_pregnant",
    "hv105": "age",
    "hv104": "sex",
    "ha0": "index_to_household",
    "ha1": "age_hemoglobin",
    "hv270": "wealth_quintile",
    "ha53": "hemoglobin_raw_adult",
    "ha56": "hemoglobin_adjusted_adult",
    "ha57": "anemia_adult",
    "hc53": "hemoglobin_raw_child",
    "hc56": "hemoglobin_adjusted_child",
    "hc57": "anemia_child",
}
hhm_data = pd.read_stata(
    directory + "IND_DHS7_2015_2016_HHM_IAPR74FL_Y2018M12D06.DTA",
    columns=hhm_columns.keys(),
)
hhm_data

CPU times: user 7.92 s, sys: 4.3 s, total: 12.2 s
Wall time: 15.4 s


,hv001,hv002,hv005,hv008,hvidx,hml18,hv105,hv104,ha0,ha1,hv270,ha53,ha56,ha57,hc53,hc56,hc57
0,10001,1,191072,1387,1,NaN,51,male,NaN,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN
1,10001,1,191072,1387,2,"not pregnant, don't know",46,female,2.0,46.0,middle,81.0,81.0,moderate,NaN,NaN,NaN
2,10001,1,191072,1387,3,NaN,22,male,NaN,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN
3,10001,1,191072,1387,4,"not pregnant, don't know",20,female,4.0,20.0,middle,113.0,113.0,mild,NaN,NaN,NaN
4,10001,9,191072,1387,1,"not pregnant, don't know",40,female,1.0,40.0,richer,116.0,116.0,mild,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2869038,360482,85,2270734,1385,1,NaN,60,male,NaN,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN
2869039,360482,85,2270734,1385,2,NaN,50,female,NaN,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN
2869040,360482,96,2270734,1385,1,NaN,66,male,NaN,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN
2869041,360482,96,2270734,1385,2,"not pregnant, don't know",46,female,2.0,46.0,middle,119.0,119.0,mild,NaN,NaN,NaN


In [15]:
hhm_data = hhm_data[hhm_columns.keys()].rename(columns=hhm_columns)

In [16]:
hhm_data["age"] = hhm_data.age.replace({"95+": 95, "don't know": np.nan}).astype(float)

In [17]:
hhm_data["sex"] = hhm_data.sex.str.title()

In [18]:
# Interesting -- sometimes age is quite off between hemoglobin and base.
hhm_data.loc[(hhm_data.age - hhm_data.age_hemoglobin).sort_values().index]

,cluster_number,household_number,weight,date_of_interview,line_number,currently_pregnant,age,sex,index_to_household,age_hemoglobin,wealth_quintile,hemoglobin_raw_adult,hemoglobin_adjusted_adult,anemia_adult,hemoglobin_raw_child,hemoglobin_adjusted_child,anemia_child
2476814,332896,13,469645,1399,4,"not pregnant, don't know",17.0,Female,4.0,48.0,richest,refused,NaN,NaN,NaN,NaN,NaN
791387,140281,68,892170,1395,4,"not pregnant, don't know",19.0,Female,4.0,49.0,middle,109.0,109.0,mild,NaN,NaN,NaN
845590,140822,66,241894,1394,4,"not pregnant, don't know",20.0,Female,4.0,49.0,poorer,119.0,108.0,mild,NaN,NaN,NaN
1062670,160904,16,2105279,1384,3,"not pregnant, don't know",19.0,Female,3.0,48.0,richest,132.0,132.0,not anemic,NaN,NaN,NaN
2709355,340097,79,339513,1382,3,"not pregnant, don't know",16.0,Female,3.0,44.0,richest,142.0,142.0,not anemic,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2869037,360482,75,2270734,1385,4,NaN,2.0,Male,NaN,NaN,richer,NaN,NaN,NaN,91.0,91.0,moderate
2869038,360482,85,2270734,1385,1,NaN,60.0,Male,NaN,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN
2869039,360482,85,2270734,1385,2,NaN,50.0,Female,NaN,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN
2869040,360482,96,2270734,1385,1,NaN,66.0,Male,NaN,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
(hhm_data.age - hhm_data.age_hemoglobin).describe()

count    749344.000000
mean         -0.041664
std           0.841376
min         -31.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          31.000000
dtype: float64

In [20]:
hhm_data["pregnant"] = hhm_data.currently_pregnant.map(
    {"not pregnant, don't know": "not_pregnant", "pregnant": "pregnant"}
)
hhm_data.loc[hhm_data.sex != "Female", "pregnant"] = "not_pregnant"
hhm_data.loc[(hhm_data.age < 15) | (hhm_data.age >= 50), "pregnant"] = "not_pregnant"
age_bin_edges = [0, 5, 15, 30, 50, 125]
age_group = pd.IntervalIndex(
    pd.cut(hhm_data.age, age_bin_edges, right=False, include_lowest=True)
)
hhm_data["age_start"] = age_group.left
hhm_data["age_end"] = age_group.right

In [21]:
hhm_data["wealth_quintile"] = data_processing.recode_dhs_wealth_quintile(
    hhm_data.wealth_quintile
)
hhm_data["weight"] = hhm_data.weight / 1_000_000

In [22]:
for type in ["child", "adult"]:
    for base_col in ["hemoglobin_raw", "hemoglobin_adjusted"]:
        col = f"{base_col}_{type}"
        hhm_data[col] = (
            hhm_data[col]
            .astype(str)
            .replace(
                {
                    "not tested": np.nan,
                    "not present": np.nan,
                    "refused": np.nan,
                    "other": np.nan,
                }
            )
            .astype(float)
        )

In [23]:
for base_col in ["hemoglobin_raw", "hemoglobin_adjusted", "anemia"]:
    assert (hhm_data.filter(like=base_col).notnull().sum(axis=1) <= 1).all()
    hhm_data[base_col] = np.nan
    # Could use bfill instead of this loop, but it was incredibly slow for me
    for col in hhm_data.filter(like=base_col).columns:
        hhm_data[base_col] = hhm_data[base_col].fillna(hhm_data[col])

In [24]:
assert (
    hhm_data[(hhm_data.sex == "male") & (hhm_data.age > 5)]
    .hemoglobin_raw.isnull()
    .all()
)

### Maternal mortality ratio

#### Maternal mortality rate

Not reported anywhere for India DHS, so we need to be extra careful since we can't cross-check.

In [25]:
adult_mortality_data.death_during_pregnancy_or_childbirth.value_counts()

no     23283
yes      615
Name: death_during_pregnancy_or_childbirth, dtype: int64

In [26]:
adult_mortality_data.death_violence_or_accident.value_counts()

no            67183
yes            7509
don't know      253
Name: death_violence_or_accident, dtype: int64

In [27]:
adult_mortality_data["age_at_death_years"] = (
    adult_mortality_data.age_at_death_unit.map(
        {"year": 1, "months": 1 / 12, "days": 1 / 365.25}
    )
    * adult_mortality_data.age_at_death
)

In [28]:
adult_mortality_data["adult_death"] = (
    adult_mortality_data.age_at_death_years >= 15
) & (adult_mortality_data.age_at_death_years < 50)

In [29]:
adult_mortality_data["date_of_death"] = adult_mortality_data.year_of_death.replace(
    {"don't know": np.nan}
).astype(float) + (
    adult_mortality_data.month_of_death.map(
        {
            "january": 1,
            "february": 2,
            "march": 3,
            "april": 4,
            "may": 5,
            "june": 6,
            "july": 7,
            "august": 8,
            "september": 9,
            "october": 10,
            "november": 11,
            "december": 12,
        }
    )
    - 0.5
) * (
    1 / 12
)

In [30]:
adult_mortality_data["date_of_birth"] = (
    adult_mortality_data.date_of_death - adult_mortality_data.age_at_death_years
)
adult_mortality_data["exposure_start"] = np.maximum(
    adult_mortality_data.date_of_birth + 15, 2011.0
)
adult_mortality_data["exposure_end"] = np.minimum(
    adult_mortality_data.date_of_birth + 50, adult_mortality_data.date_of_death
)
adult_mortality_data["exposure"] = np.maximum(
    adult_mortality_data.exposure_end - adult_mortality_data.exposure_start, 0
)
adult_mortality_data["weighted_exposure"] = (
    adult_mortality_data.weight * adult_mortality_data.exposure
)
adult_mortality_data

/ihme/homes/ndbs/miniconda3/envs/vivarium_gates_lsff_2026_artifact/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in maximum
  result = getattr(ufunc, method)(*inputs, **kwargs)
/ihme/homes/ndbs/miniconda3/envs/vivarium_gates_lsff_2026_artifact/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in minimum
  result = getattr(ufunc, method)(*inputs, **kwargs)
/ihme/homes/ndbs/miniconda3/envs/vivarium_gates_lsff_2026_artifact/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in maximum
  result = getattr(ufunc, method)(*inputs, **kwargs)


,weight,any_died,num_died,wealth_quintile,sex,age_at_death_unit,age_at_death,month_of_death,year_of_death,death_violence_or_accident,death_during_pregnancy_or_childbirth,age_at_death_years,adult_death,date_of_death,date_of_birth,exposure_start,exposure_end,exposure,weighted_exposure
0,0.008939,yes,1.0,3,male,year,51.0,april,2015.0,no,NaN,51.0,False,2015.291667,1964.291667,2011.0,2014.291667,3.291667,0.029424
1,0.008939,yes,1.0,3,male,months,56.0,april,2015.0,yes,NaN,4.666667,False,2015.291667,2010.625,2025.625,2015.291667,0,0.0
2,0.004495,yes,1.0,3,male,year,45.0,august,2014.0,no,NaN,45.0,True,2014.625000,1969.625,2011.0,2014.625,3.625,0.016294
3,0.004495,yes,2.0,4,male,year,35.0,may,2014.0,no,NaN,35.0,True,2014.375000,1979.375,2011.0,2014.375,3.375,0.015171
4,0.004495,yes,2.0,4,male,year,40.0,august,2014.0,no,NaN,40.0,True,2014.625000,1974.625,2011.0,2014.625,3.625,0.016294
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74940,2.463153,yes,2.0,2,female,year,21.0,september,2013.0,yes,NaN,21.0,True,2013.708333,1992.708333,2011.0,2013.708333,2.708333,6.671039
74941,2.270734,yes,1.0,3,female,year,80.0,november,2014.0,no,no,80.0,False,2014.875000,1934.875,2011.0,1984.875,0,0.0
74942,2.270734,yes,1.0,3,male,year,50.0,january,2015.0,no,NaN,50.0,False,2015.041667,1965.041667,2011.0,2015.041667,4.041667,9.17755
74943,2.270734,yes,1.0,3,male,months,45.0,january,2012.0,yes,NaN,3.75,False,2012.041667,2008.291667,2023.291667,2012.041667,0,0.0


In [31]:
living_exposure_data = hhm_data.copy()
living_exposure_data["date_of_birth"] = (
    (living_exposure_data.date_of_interview / 12) + 1900 - living_exposure_data.age
)
living_exposure_data["exposure_start"] = np.maximum(
    living_exposure_data.date_of_birth + 15, 2011.0
)
living_exposure_data["exposure_end"] = np.minimum(
    living_exposure_data.date_of_birth + 50,
    (living_exposure_data.date_of_interview / 12) + 1900,
)
living_exposure_data["exposure"] = np.maximum(
    living_exposure_data.exposure_end - living_exposure_data.exposure_start, 0
)
living_exposure_data["weighted_exposure"] = (
    living_exposure_data.weight * living_exposure_data.exposure
)
living_exposure_data

,cluster_number,household_number,weight,date_of_interview,line_number,currently_pregnant,age,sex,index_to_household,age_hemoglobin,...,age_start,age_end,hemoglobin_raw,hemoglobin_adjusted,anemia,date_of_birth,exposure_start,exposure_end,exposure,weighted_exposure
0,10001,1,0.191072,1387,1,NaN,51.0,Male,NaN,NaN,...,50.0,125.0,NaN,NaN,NaN,1964.583333,2011.0,2014.583333,3.583333,0.684675
1,10001,1,0.191072,1387,2,"not pregnant, don't know",46.0,Female,2.0,46.0,...,30.0,50.0,81.0,81.0,moderate,1969.583333,2011.0,2015.583333,4.583333,0.875747
2,10001,1,0.191072,1387,3,NaN,22.0,Male,NaN,NaN,...,15.0,30.0,NaN,NaN,NaN,1993.583333,2011.0,2015.583333,4.583333,0.875747
3,10001,1,0.191072,1387,4,"not pregnant, don't know",20.0,Female,4.0,20.0,...,15.0,30.0,113.0,113.0,mild,1995.583333,2011.0,2015.583333,4.583333,0.875747
4,10001,9,0.191072,1387,1,"not pregnant, don't know",40.0,Female,1.0,40.0,...,30.0,50.0,116.0,116.0,mild,1975.583333,2011.0,2015.583333,4.583333,0.875747
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2869038,360482,85,2.270734,1385,1,NaN,60.0,Male,NaN,NaN,...,50.0,125.0,NaN,NaN,NaN,1955.416667,2011.0,2005.416667,0.000000,0.000000
2869039,360482,85,2.270734,1385,2,NaN,50.0,Female,NaN,NaN,...,50.0,125.0,NaN,NaN,NaN,1965.416667,2011.0,2015.416667,4.416667,10.029075
2869040,360482,96,2.270734,1385,1,NaN,66.0,Male,NaN,NaN,...,50.0,125.0,NaN,NaN,NaN,1949.416667,2011.0,1999.416667,0.000000,0.000000
2869041,360482,96,2.270734,1385,2,"not pregnant, don't know",46.0,Female,2.0,46.0,...,30.0,50.0,119.0,119.0,mild,1969.416667,2011.0,2015.416667,4.416667,10.029075


In [32]:
(adult_mortality_data.adult_death * adult_mortality_data.weight).sum()

11897.673413

In [33]:
adult_mortality_data.weighted_exposure.sum()

42011.871167166595

In [34]:
((adult_mortality_data.adult_death * adult_mortality_data.weight).sum()) / (
    adult_mortality_data.weighted_exposure.sum()
    + living_exposure_data.weighted_exposure.sum()
)

0.0017707888221588613

In [35]:
adult_mortality_data

,weight,any_died,num_died,wealth_quintile,sex,age_at_death_unit,age_at_death,month_of_death,year_of_death,death_violence_or_accident,death_during_pregnancy_or_childbirth,age_at_death_years,adult_death,date_of_death,date_of_birth,exposure_start,exposure_end,exposure,weighted_exposure
0,0.008939,yes,1.0,3,male,year,51.0,april,2015.0,no,NaN,51.0,False,2015.291667,1964.291667,2011.0,2014.291667,3.291667,0.029424
1,0.008939,yes,1.0,3,male,months,56.0,april,2015.0,yes,NaN,4.666667,False,2015.291667,2010.625,2025.625,2015.291667,0,0.0
2,0.004495,yes,1.0,3,male,year,45.0,august,2014.0,no,NaN,45.0,True,2014.625000,1969.625,2011.0,2014.625,3.625,0.016294
3,0.004495,yes,2.0,4,male,year,35.0,may,2014.0,no,NaN,35.0,True,2014.375000,1979.375,2011.0,2014.375,3.375,0.015171
4,0.004495,yes,2.0,4,male,year,40.0,august,2014.0,no,NaN,40.0,True,2014.625000,1974.625,2011.0,2014.625,3.625,0.016294
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74940,2.463153,yes,2.0,2,female,year,21.0,september,2013.0,yes,NaN,21.0,True,2013.708333,1992.708333,2011.0,2013.708333,2.708333,6.671039
74941,2.270734,yes,1.0,3,female,year,80.0,november,2014.0,no,no,80.0,False,2014.875000,1934.875,2011.0,1984.875,0,0.0
74942,2.270734,yes,1.0,3,male,year,50.0,january,2015.0,no,NaN,50.0,False,2015.041667,1965.041667,2011.0,2015.041667,4.041667,9.17755
74943,2.270734,yes,1.0,3,male,months,45.0,january,2012.0,yes,NaN,3.75,False,2012.041667,2008.291667,2023.291667,2012.041667,0,0.0


In [36]:
adult_mortality_data["maternal_death"] = (
    adult_mortality_data.adult_death
    & (adult_mortality_data.death_during_pregnancy_or_childbirth == "yes")
    & (adult_mortality_data.death_violence_or_accident != "yes")
)

In [37]:
(adult_mortality_data.maternal_death * adult_mortality_data.weight).sum()

405.651753

In [38]:
def maternal_mortality_rate(df):
    return ((df.maternal_death * df.weight).sum() * 1_000) / (
        df.weighted_exposure
    ).sum()

In [39]:
maternal_mortality_data = pd.concat(
    [
        adult_mortality_data[adult_mortality_data.sex == "female"][
            ["wealth_quintile", "maternal_death", "weight", "weighted_exposure"]
        ],
        living_exposure_data[living_exposure_data.sex == "female"][
            ["wealth_quintile", "weight", "weighted_exposure"]
        ].assign(maternal_death=False),
    ],
    ignore_index=True,
)

In [40]:
maternal_mortality_rate(maternal_mortality_data)

24.949186747614135

In [41]:
maternal_mortality_rates = maternal_mortality_data.groupby("wealth_quintile").apply(
    maternal_mortality_rate
)
maternal_mortality_rates

wealth_quintile
1    36.804365
2    21.447157
3    27.064862
4    21.697309
5    11.040954
dtype: float64

#### General fertility rate

In [42]:
fertility_event_data = birth_data.copy()
fertility_event_data["birth_in_period"] = (
    (fertility_event_data.interview_date - fertility_event_data.birth_date) >= 1
) & ((fertility_event_data.interview_date - fertility_event_data.birth_date) <= 36)
fertility_event_data["weighted_birth_in_period"] = (
    fertility_event_data.birth_in_period * fertility_event_data.weight
)

In [43]:
fertility_event_data.weighted_birth_in_period.sum()

150433.10539200003

In [44]:
fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum()

wealth_quintile
1    37315.912347
2    33131.370611
3    30243.631415
4    27530.819389
5    22211.371630
Name: weighted_birth_in_period, dtype: float64

In [45]:
fertility_exposure_data = wra_data.copy()
fertility_exposure_data["exposure_start"] = np.maximum(
    fertility_exposure_data.date_of_birth + 12 * 15,
    fertility_exposure_data.interview_date - 36,
)  # aka lowlim
# aka upplim
fertility_exposure_data["exposure_end"] = np.minimum(
    fertility_exposure_data.interview_date - 1,
    fertility_exposure_data.date_of_birth + 12 * 45,
)
fertility_exposure_data["exposure"] = (
    (fertility_exposure_data.exposure_end - fertility_exposure_data.exposure_start) + 1
).clip(lower=0)
fertility_exposure_data["weighted_exposure"] = (
    fertility_exposure_data.exposure * fertility_exposure_data.weight
)

In [46]:
gfr_by_wealth = (
    fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum()
    * 1_000
    / (fertility_exposure_data.groupby("wealth_quintile").weighted_exposure.sum() / 12)
)
gfr_by_wealth

wealth_quintile
1    114.405014
2     91.556480
3     79.049536
4     69.588896
5     56.835044
dtype: float64

In [47]:
maternal_disorders_incidence_disparities = (
    (maternal_mortality_rates / gfr_by_wealth)
    .rename("value")
    .rename_axis("wealth_quintile")
    .reset_index()
)
maternal_disorders_incidence_disparities.insert(0, "sex", "Female")
maternal_disorders_incidence_disparities

,sex,wealth_quintile,value
0,Female,1,0.321702
1,Female,2,0.234251
2,Female,3,0.342379
3,Female,4,0.311793
4,Female,5,0.194263


In [48]:
maternal_disorders_incidence_disparities.to_csv(
    f"{results_dir}/maternal_disorders_incidence_disparities/india.csv",
    index=False,
)